In [9]:
# Importing functions
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

# 1. Data Collection
Supermarket location data was collected from OpenStreetMap using Overpass Turbo.

The dataset contains 72 supermarket records within the selected map area in Perth. Each record includes geographic information identifying where a supermarket is located.

Since the data was collected based on the selected map area, it may not cover all of Greater Perth. Geographic coverage will need to be verified before conducting the accessibility analysis.

In [10]:
# Read geojson file and storing it into a variable
stores = gpd.read_file("../data/raw/perth_supermarkets.geojson")
# Check the variables in the file
print(f"Total supermarket records: {len(stores)}")
print(stores.columns.tolist())
# Check the header of the dataset
stores.head()

Total supermarket records: 470
['id', '@id', 'access', 'addr:floor', 'addr:housenumber', 'addr:postcode', 'addr:state', 'addr:street', 'addr:suburb', 'addr:unit', 'air_conditioning', 'alt_name', 'branch', 'branch:ref', 'branch:website', 'brand', 'brand:phone', 'brand:website', 'brand:wikidata', 'brand:wikipedia', 'building', 'building:levels', 'building:part', 'changing_table', 'check_date', 'check_date:opening_hours', 'contact:email', 'contact:facebook', 'contact:instagram', 'contact:phone', 'contact:website', 'cuisine', 'currency:AUD', 'description', 'email', 'facebook', 'fax', 'fixme', 'full_name', 'indoor', 'internet_access', 'internet_access:fee', 'landuse', 'layer', 'level', 'loc_name', 'logo', 'long_name', 'name', 'name:en', 'name:zh', 'nat_name', 'not:brand:wikidata', 'note', 'official_name', 'opening_hours', 'opening_hours:signed', 'operator', 'operator:ref:abn', 'operator:ref:acn', 'operator:type', 'operator:website', 'operator:wikidata', 'organic', 'origin', 'payment:all', '

,id,@id,access,addr:floor,addr:housenumber,addr:postcode,addr:state,addr:street,addr:suburb,addr:unit,...,survey:date,toilets,toilets:menstrual_products,toilets:wheelchair,website,wheelchair,wholesale,@geometry,zero_waste,geometry
0,way/25635449,way/25635449,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,center,NaN,POINT (115.83858 -31.83878)
1,way/32052482,way/32052482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,center,NaN,POINT (115.7575 -32.07084)
2,way/47078785,way/47078785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,center,NaN,POINT (115.82023 -31.94301)
3,way/47435394,way/47435394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,center,NaN,POINT (115.85047 -32.01154)
4,way/48536401,way/48536401,NaN,NaN,38,NaN,NaN,Victoria Street,NaN,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,center,NaN,POINT (116.00395 -31.89091)


# 2. Data Profiling and Assessment

In [11]:
column_summary = pd.DataFrame({
    "Non_Null": stores.notna().sum(),
    "Missing": stores.isna().sum(),
    "Missing_Percent": stores.isna().mean() * 100
})

column_summary = column_summary.sort_values(
    "Non_Null", ascending=False
)

display(column_summary.head(30))

,Non_Null,Missing,Missing_Percent
id,470,0,0.000000
@id,470,0,0.000000
shop,470,0,0.000000
geometry,470,0,0.000000
name,460,10,2.127660
brand,318,152,32.340426
brand:wikidata,317,153,32.553191
operator,219,251,53.404255
brand:wikipedia,178,292,62.127660
opening_hours,176,294,62.553191


In [12]:
print("Store classifications:")
print(stores["shop"].value_counts(dropna=False))

print("\nTop supermarket brands:")
print(stores["brand"].value_counts(dropna=False).head(15))

print("\nMissing store names:")
print(stores["name"].isna().sum())

Store classifications:
shop
supermarket    470
Name: count, dtype: int64

Top supermarket brands:
brand
NaN                      152
IGA                       92
Woolworths                86
Coles                     79
Aldi                      27
ALDI                      16
Spudshed                  11
Foodworks                  2
Coles Local                2
Campbells                  1
NP                         1
The Source Bulk Foods      1
Name: count, dtype: int64

Missing store names:
10


In [13]:
print("Duplicate OSM IDs:")
print(stores["@id"].duplicated().sum())

print("\nGeometry types:")
print(stores.geometry.geom_type.value_counts())

print("\nMissing geometries:")
print(stores.geometry.isna().sum())

print("\nEmpty geometries:")
print(stores.geometry.is_empty.sum())

print("\nInvalid geometries:")
print((~stores.geometry.is_valid).sum())

print("\nCoordinate Reference System:")
print(stores.crs)

Duplicate OSM IDs:
0

Geometry types:
Point    470
Name: count, dtype: int64

Missing geometries:
0

Empty geometries:
0

Invalid geometries:
0

Coordinate Reference System:
EPSG:4326


In [14]:
# Retain only attributes needed for accessibility analysis
stores_clean = stores[["@id", "geometry"]].copy()

# Convert to latitude/longitude
stores_clean = stores_clean.to_crs(epsg=4326)

# Standardise geometries as points
stores_clean["geometry"] = (
    stores_clean.geometry.representative_point()
)

# Rename identifier
stores_clean = stores_clean.rename(
    columns={"@id": "store_id"}
)

stores_clean.head()

# Saving file
stores_clean.to_file(
    "../data/processed/perth_supermarkets_clean.geojson",
    driver="GeoJSON"
)

The supermarket dataset was examined to determine its suitability for identifying areas with limited grocery accessibility.

The dataset contains 72 supermarket records, all with valid geographic information. No duplicate OSM identifiers, missing geometries, or invalid geometries were identified. The records consist of 58 points and 14 polygons, which can be standardised into point locations for spatial analysis.

Since the project's primary objective is to measure supermarket accessibility, attributes such as brand, opening hours, and payment methods are not required. Only geographic locations and store identifiers were retained.

However, the geographic coverage and completeness of the supermarket dataset must be verified before conducting accessibility analysis, as missing stores could incorrectly classify neighbourhoods as underserved.

In [15]:
# Ensure coordinates are in latitude/longitude
stores = stores.to_crs(epsg=4326)

# Convert polygon geometries into points
store_points = stores.copy()
store_points["geometry"] = stores.geometry.representative_point()

# Create a map centred on Perth
m = folium.Map(
    location=[-31.95, 115.86],
    zoom_start=11
)

# Plot supermarket locations
for _, row in store_points.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        popup=str(row.get("name", "Supermarket"))
    ).add_to(m)

m

In [16]:
m.save("../outputs/maps/perth_supermarkets.html")

Supermarket locations were standardised into geographic points, as some stores were originally represented by building boundaries rather than individual coordinates.

These points were then plotted on an interactive map to visualise the distribution of supermarkets within the selected area of Perth.

The processed location data will be used for subsequent spatial analysis to identify areas with limited supermarket accessibility.